### Movie Recommendation System 

In [ ]:
import pandas as pd 

In [ ]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')
new_movies = pd.read_csv('telugu_movies_tmdb_format.csv')

# Cast/crew for the Telugu films only exist after running enrich_telugu_from_tmdb.py
import os
new_credits = pd.read_csv('telugu_credits_tmdb_format.csv') if os.path.exists('telugu_credits_tmdb_format.csv') else None


In [ ]:
movies.head(3)

In [ ]:
credits.head(3)

In [ ]:
credits['cast'][0]

In [ ]:
import re
norm = lambda s: re.sub(r'[^a-z0-9]', '', str(s).lower())

# Skip Telugu rows whose title is already in the Kaggle data (e.g. Baahubali) so no movie appears twice
new_movies = new_movies[~new_movies['title'].map(norm).isin(movies['title'].map(norm))]

# Stack the Telugu rows UNDER the Kaggle rows (same 20 columns). Do not merge them on title.
all_movies = pd.concat([movies, new_movies], ignore_index=True).drop_duplicates('id')

parts = [credits.rename(columns={'movie_id': 'id'})[['id', 'cast', 'crew']]]
if new_credits is not None:
    parts.append(new_credits.rename(columns={'movie_id': 'id'})[['id', 'cast', 'crew']])
all_credits = pd.concat(parts, ignore_index=True).drop_duplicates('id')

# Join on id (titles repeat, which creates extra rows); 'left' keeps Telugu films that have no credits yet
data = all_movies.merge(all_credits, on='id', how='left')
data[['cast', 'crew']] = data[['cast', 'crew']].fillna('[]')


In [ ]:
data.shape

In [ ]:
data.head(3)

In [ ]:
data.info()

In [ ]:
data['keywords'][0]

In [ ]:
data = data[['genres','id','keywords','title','release_date','overview','cast','crew']]

In [ ]:
data.info()

In [ ]:
data.dropna(subset=['overview'], inplace=True)
# IMPORTANT: row position must match the similarity matrix, so renumber 0..n-1 after dropping rows
data = data.reset_index(drop=True)


In [ ]:
data.isnull().sum()

In [ ]:
data.head(3)

In [ ]:
data['genres'][0]

In [ ]:
import ast

In [ ]:
def genre_extract(col):
    L=[]
    for i in ast.literal_eval(col):
        L.append(i['name'])
    return L


In [ ]:
data['genres']=data['genres'].apply(genre_extract)

In [ ]:
data.head(3)

In [ ]:
data['cast']

In [ ]:
data['keywords'][0]

In [ ]:
def keyword_extract(col):
    L=[]
    for i in ast.literal_eval(col):
        L.append(i['name'])
    return L
data['keywords']=data['keywords'].apply(keyword_extract)

In [ ]:
data['cast']

In [ ]:
def overview_extract(col):
    l=[]
    for i in col.split():
        l.append(i)
    return l
data['overview']=data['overview'].apply(overview_extract)

In [ ]:
data['cast']

In [ ]:
def cast_extract(col):
    L=[]
    j=1
    for i in ast.literal_eval(col):
        if j<=3:
            L.append(i['name'])
            j+=1
    return L
data['cast']=data['cast'].apply(cast_extract)

In [ ]:
data.head(3)

In [ ]:
data['crew'][0]

In [ ]:
def crew_extract(col):
    L=[]
    for i in ast.literal_eval(col):
        if(i['job']=='Director'):
            L.append(i['name'])
            break
    return L
data['crew']=data['crew'].apply(crew_extract)
        

In [ ]:
data.head(3)

In [ ]:
data['cast'] = data['cast'].apply(lambda x: [i.replace(" ", "") for i in x])
data['crew'] = data['crew'].apply(lambda x: [i.replace(" ", "") for i in x])
data['keywords'] = data['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])
data['genres'] = data['genres'].apply(lambda x: [i.replace(" ", "") for i in x])
data['overview'] = data['overview'].apply(lambda x: [i.replace(" ", "") for i in x]) 
data.head(3)

In [ ]:
data['title']=data['title'].apply(lambda x:x.lower())

# Some titles repeat. First add the year, then number any that are still identical,
# so every title can be selected in the app.
year = data['release_date'].fillna('').str[:4]
dup = data['title'].duplicated(keep=False) & (year != '')
data.loc[dup, 'title'] = data.loc[dup, 'title'] + ' (' + year[dup] + ')'
n = data.groupby('title').cumcount()
data.loc[n > 0, 'title'] = data.loc[n > 0, 'title'] + ' [' + (n[n > 0] + 1).astype(str) + ']'

In [ ]:
data['tag']=data['overview']+data['genres']+data['keywords']+data['overview']+data['cast']+data['crew']

In [ ]:
data.head(3)

In [ ]:
data.drop(columns=['genres', 'keywords', 'release_date', 'overview', 'cast', 'crew'], inplace=True)

In [ ]:
data.head(3)

In [ ]:
data['tag']=data['tag'].str.join(" ")

In [ ]:
data['tag']=data['tag'].apply(lambda x:x.lower())

In [ ]:
data.head(3)

In [ ]:
pip install nltk

In [ ]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [ ]:
def stemming(col):
    return " ".join(ps.stem(word) for word in col.split())

In [ ]:
data['tag']=data['tag'].apply(stemming)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(max_features=5000,stop_words='english')

In [ ]:
vectors=cv.fit_transform(data['tag'])   # keep it sparse: far less memory than .toarray()

In [ ]:
vectors

In [ ]:
cv.get_feature_names_out()[:200]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
similarities=cosine_similarity(vectors).astype('float32')   # float32 halves the file size

In [ ]:
similarities

In [ ]:
def recomend(movie):
    movie_index=data[data['title']==movie].index[0]
    recomendation=similarities[movie_index]

    movies_list=sorted(enumerate(recomendation),reverse=True,key=lambda x:x[1])[1:6]
    for i in movies_list:
        print(data.iloc[i[0]].title)

In [ ]:
recomend('avatar')

#### Try a Telugu movie

In [ ]:
telugu_titles = data[data['title'].str.contains('baahubali|rrr|nenokkadine', regex=True)]['title'].tolist()
print(telugu_titles)
recomend(telugu_titles[0])

#### Save the files used by the Streamlit app

In [ ]:
import pickle, joblib
assert data.index.equals(pd.RangeIndex(len(data))), "index must be 0..n-1"
assert similarities.shape[0] == len(data)

with open('movies.pkl', 'wb') as f:
    pickle.dump(data, f)
joblib.dump(similarities, 'similarity.joblib')
print(len(data), 'movies saved')